# Comprensión y análisis exploratorio de datos - Proyecto Integrador M5

En este notebook se realiza la comprensión inicial y el análisis exploratorio de la base de datos del Proyecto Integrador M5.

El objetivo es analizar la estructura del dataset, identificar la variable objetivo, revisar valores faltantes, estudiar la distribución de las variables y detectar posibles inconsistencias que deban considerarse antes de avanzar hacia etapas de limpieza, ingeniería de variables y modelado.

La base corresponde a información crediticia y financiera de clientes, con variables asociadas al préstamo, perfil del cliente, comportamiento crediticio y estado de pago.

La variable objetivo definida para el proyecto es `Pago_atiempo`, la cual indica si el cliente realizó el pago a tiempo.

In [1]:
# ============================================================
# 1. Importación de librerías y configuración inicial
# ============================================================
import json
from pathlib import Path
import pandas as pd
import numpy as np
# ============================================================
# 2. Lectura del archivo de configuración
# ============================================================
ruta_actual = Path.cwd()
ruta_config = ruta_actual / "src" / "config.json"
if not ruta_config.exists():
    ruta_config = ruta_actual / "config.json"
with open(ruta_config, "r", encoding="utf-8") as archivo:
    config = json.load(archivo)
# ============================================================
# 3. Carga de la base de datos
# ============================================================
ruta_base = (ruta_config.parent / config["base_datos"]).resolve()
df = pd.read_excel(ruta_base)
# ============================================================
# 4. Definición de variable objetivo
# ============================================================
target = config["target"]
# ============================================================
# 5. Verificación inicial
# ============================================================
print("Base cargada correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Variable objetivo definida: {target}")
display(df.head())

Base cargada correctamente.
Filas: 10763
Columnas: 23
Variable objetivo definida: Pago_atiempo


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


In [ ]:
# ============================================================
# 6. Estructura general del dataset
# ============================================================
resumen_estructura = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique().values,
    "nulos": df.isnull().sum().values,
    "porcentaje_nulos": (df.isnull().mean().values * 100).round(2)
})
resumen_estructura

,columna,tipo_dato,valores_unicos,nulos,porcentaje_nulos
0,tipo_credito,int64,6,0,0.00
1,fecha_prestamo,datetime64[us],10758,0,0.00
2,capital_prestado,float64,7306,0,0.00
3,plazo_meses,int64,18,0,0.00
4,edad_cliente,int64,54,0,0.00
5,tipo_laboral,str,2,0,0.00
6,salario_cliente,int64,1385,0,0.00
7,total_otros_prestamos,int64,1538,0,0.00
8,cuota_pactada,int64,9836,0,0.00
9,puntaje,float64,248,0,0.00


In [3]:
# ============================================================
# 7. Distribución de la variable objetivo
# ============================================================
distribucion_target = (
    df[target]
    .value_counts(dropna=False)
    .reset_index()
)
distribucion_target.columns = [target, "cantidad"]
distribucion_target["porcentaje"] = (
    distribucion_target["cantidad"] / len(df) * 100
).round(2)
display(distribucion_target)

,Pago_atiempo,cantidad,porcentaje
0,1,10252,95.25
1,0,511,4.75


## Distribución de la variable objetivo

La variable objetivo del proyecto es `Pago_atiempo`.

En la base analizada, el **95,25%** de los registros corresponde a clientes que pagaron a tiempo (`Pago_atiempo = 1`), mientras que solo el **4,75%** corresponde a clientes que no pagaron a tiempo (`Pago_atiempo = 0`).

Esto indica un **desbalance significativo de clases**, ya que la clase positiva domina ampliamente el dataset. Esta situación deberá tenerse en cuenta en las etapas de modelado, dado que una métrica como `accuracy` podría resultar engañosa si el modelo aprende a predecir mayoritariamente la clase más frecuente.

Por este motivo, en la evaluación del modelo será necesario considerar métricas complementarias como `precision`, `recall`, `f1-score`, matriz de confusión y ROC-AUC.